In [2]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

#
# Change "Questionnnair" -> "Questionnaire"
# if your actual folder uses the correct spelling.
#
DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "ACQ_J.xpt"
csv_file = DATA_DIR / "ACQ_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2017-2018 ACQ_J XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "ACD011A",
    "ACD011B",
    "ACD011C",
    "ACD040",
    "ACD110"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "ACQ_J column names/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 8421:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 6:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 8,421 rows x 6 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of attached ACQ_J:
#
#     string fields             = NONE
#     genuine decimal fields    = NONE
#
#
# All six variables contain only:
#
#     SEQN -> respondent ID
#     categorical language codes
#
#
# Therefore all six can safely become nullable Int64
# after proving every populated value is whole-number.
#
#
# IMPORTANT:
#
# This applies ONLY to ACQ_J.
#
# Your broader NHANES fidelity rule remains:
#
#     Dietary decimals
#     laboratory continuous values
#     examination measurements
#     survey/sample weights
#
# must remain exactly AS IS.
#

STRING_COLUMNS = []

DECIMAL_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 6:

    raise ValueError(
        "Expected exactly 6 whole-number fields."
    )


print(
    "\n--- FIELD TYPE PLAN ---"
)

print(
    "Character fields:",
    STRING_COLUMNS
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN SAS XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT representation of numeric zero:
#
#     5.397605346934028e-79
#
#
# Attached ACQ_J contains NONE.
#
# Therefore NO tiny -> zero repair is required.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 6 ACQ_J variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


tiny_variable_count = int(
    tiny_counts.gt(0)
    .sum()
)


ordinary_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


ordinary_zero_total = int(
    ordinary_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)

print(
    "Ordinary numeric zeros:",
    f"{ordinary_zero_total:,}"
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in ACQ_J."
    )


if ordinary_zero_total != 0:

    print(
        ordinary_zero_counts[
            ordinary_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected ordinary numeric zeros "
        "found in ACQ_J."
    )


print(
    "PASS: ACQ_J contains 0 tiny artifacts "
    "and 0 ordinary numeric zeros."
)


# ============================================================
# 6. NO ZERO RESTORATION REQUIRED
# ============================================================

#
# Since ACQ_J contains no XPORT zero corruption,
# source_df itself is the exact fidelity reference.
#

expected_df = source_df.copy()


print(
    "\nPASS: No tiny-to-zero repair required."
)


# ============================================================
# 7. DETECT GENUINE FRACTIONAL VARIABLES
# ============================================================

#
# Run BEFORE Int64 conversion.
#
# np.round() here is used ONLY as a diagnostic.
#
# It does not modify source data.
#

print(
    "\n--- FRACTIONAL VALUE DETECTION ---"
)


fractional_columns = []

fractional_counts = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = count


    print(
        f"{col}: "
        f"{count:,} fractional observations"
    )


    if count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected fractional variables found.\n\n"
        "These values will NOT be rounded:\n"
        + str(
            fractional_columns
        )
    )


print(
    "PASS: No genuine decimal/fractional "
    "variables found in ACQ_J."
)


# ============================================================
# 8. VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


print(
    "\n--- SEQN BEFORE CLEANING ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93705:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 93705-102956 validated."
)


# ============================================================
# 9. PROVE ALL SIX VARIABLES ARE WHOLE-NUMBER
# ============================================================

whole_number_failures = []


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if not whole_mask.all():

        whole_number_failures.append(
            col
        )


if whole_number_failures:

    raise ValueError(
        "Unexpected fractional values found in:\n"
        + str(
            whole_number_failures
        )
    )


print(
    "PASS: All 6 ACQ_J variables are "
    "mathematically whole-number."
)


# ============================================================
# 10. CONVERT ALL SIX VARIABLES TO NULLABLE Int64
# ============================================================

#
# NO .round() is used here.
#
# We already proved the values are whole-number.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: All 6 fields converted to nullable Int64 "
    "without rounding."
)


# ============================================================
# 11. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed frequencies:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 12. CDC CHECK - ACD011A
# ============================================================

#
# Speak English at home.
#
# CDC:
#
#     1 = English       5,230
#     77 = Refused          0
#     99 = Don't know       0
#
# Missing = 3,191
#

check_frequency(
    "ACD011A",
    {
        1: 5230
    },
    3191
)


# ============================================================
# 13. CDC CHECK - ACD011B
# ============================================================

#
# Speak Spanish at home.
#
#     8 = Spanish
#
# Count = 16
# Missing = 8,405
#

check_frequency(
    "ACD011B",
    {
        8: 16
    },
    8405
)


# ============================================================
# 14. CDC CHECK - ACD011C
# ============================================================

#
# Speak other language at home.
#
#     9 = Other
#
# Count = 159
# Missing = 8,262
#

check_frequency(
    "ACD011C",
    {
        9: 159
    },
    8262
)


# ============================================================
# 15. VALIDATE ACD011 MULTI-RESPONSE CODES
# ============================================================

#
# ACD011A/B/C are "check all that apply".
#
# Their released nonmissing codes should therefore be:
#
#     ACD011A -> 1
#     ACD011B -> 8
#     ACD011C -> 9
#

EXPECTED_ACD011_CODES = {
    "ACD011A": 1,
    "ACD011B": 8,
    "ACD011C": 9
}


print(
    "\n--- ACD011 MULTI-RESPONSE CODE CHECK ---"
)


for col, expected_code in EXPECTED_ACD011_CODES.items():

    observed_codes = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )


    print(
        f"{col}:",
        observed_codes
    )


    if observed_codes != {expected_code}:

        raise ValueError(
            f"{col} contains unexpected codes: "
            f"{observed_codes}"
        )


print(
    "PASS: ACD011A/B/C contain only "
    "their designated response codes."
)


# ============================================================
# 16. CDC CHECK - ACD040
# ============================================================

#
# Language(s) spoken at home - Hispanic participants:
#
#     1 = Only Spanish                  529
#     2 = More Spanish than English     322
#     3 = Both equally                  334
#     4 = More English than Spanish     353
#     5 = Only English                  414
#     7 = Refused                         2
#     9 = Don't know                      3
#
# Missing = 6,464
#

check_frequency(
    "ACD040",
    {
        1: 529,
        2: 322,
        3: 334,
        4: 353,
        5: 414,
        7: 2,
        9: 3
    },
    6464
)


# ============================================================
# 17. CDC CHECK - ACD110
# ============================================================

#
# Language(s) spoken at home - Asian participants:
#
#     1 = Only non-English language          439
#     2 = More non-English than English      107
#     3 = Both equally                       124
#     4 = More English than non-English      134
#     5 = Only English                       327
#
# CDC has zero observed Refused / Don't know.
#
# Missing = 7,290
#

check_frequency(
    "ACD110",
    {
        1: 439,
        2: 107,
        3: 124,
        4: 134,
        5: 327
    },
    7290
)


# ============================================================
# 18. VALID RESPONSE CODE CHECK
# ============================================================

VALID_CODES = {

    "ACD011A": {
        1
    },

    "ACD011B": {
        8
    },

    "ACD011C": {
        9
    },

    "ACD040": {
        1, 2, 3, 4, 5, 7, 9
    },

    "ACD110": {
        1, 2, 3, 4, 5
    }
}


print(
    "\n--- VALID RESPONSE CODE CHECK ---"
)


for col, allowed_codes in VALID_CODES.items():

    observed_codes = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )


    unexpected_codes = (
        observed_codes
        -
        allowed_codes
    )


    print(
        f"{col}:",
        sorted(
            observed_codes
        )
    )


    if unexpected_codes:

        raise ValueError(
            f"{col} contains unexpected codes: "
            f"{sorted(unexpected_codes)}"
        )


print(
    "PASS: No unexpected ACQ response codes."
)


# ============================================================
# 19. LANGUAGE QUESTION GROUP STRUCTURE
# ============================================================

#
# CDC routes participants to ONE of three question groups
# based on self-identified race/Hispanic origin:
#
# 1) ACD011A/B/C
#
# 2) ACD040
#
# 3) ACD110
#
#
# Since DEMO/RIDRETH3 is not in ACQ_J itself,
# we cannot independently recreate the race routing.
#
# But we CAN confirm the released response groups
# never overlap.
#

acd011_present = (
    df[
        [
            "ACD011A",
            "ACD011B",
            "ACD011C"
        ]
    ]
    .notna()
    .any(axis=1)
    .to_numpy(
        dtype=bool
    )
)


acd040_present = (
    df["ACD040"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


acd110_present = (
    df["ACD110"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


acd011_count = int(
    acd011_present.sum()
)


acd040_count = int(
    acd040_present.sum()
)


acd110_count = int(
    acd110_present.sum()
)


overlap_011_040 = int(
    np.sum(
        acd011_present
        &
        acd040_present
    )
)


overlap_011_110 = int(
    np.sum(
        acd011_present
        &
        acd110_present
    )
)


overlap_040_110 = int(
    np.sum(
        acd040_present
        &
        acd110_present
    )
)


any_language_response = (
    acd011_present
    |
    acd040_present
    |
    acd110_present
)


no_language_response_count = int(
    (
        ~any_language_response
    )
    .sum()
)


print(
    "\n--- ACQ QUESTION-GROUP STRUCTURE ---"
)

print(
    "ACD011A/B/C group:",
    f"{acd011_count:,}"
)

print(
    "ACD040 group:",
    f"{acd040_count:,}"
)

print(
    "ACD110 group:",
    f"{acd110_count:,}"
)

print(
    "ACD011 vs ACD040 overlap:",
    overlap_011_040
)

print(
    "ACD011 vs ACD110 overlap:",
    overlap_011_110
)

print(
    "ACD040 vs ACD110 overlap:",
    overlap_040_110
)

print(
    "No language-response variable populated:",
    no_language_response_count
)


if acd011_count != 5308:

    raise ValueError(
        "Unexpected ACD011 response-group count."
    )


if acd040_count != 1957:

    raise ValueError(
        "Unexpected ACD040 response-group count."
    )


if acd110_count != 1131:

    raise ValueError(
        "Unexpected ACD110 response-group count."
    )


if overlap_011_040 != 0:

    raise ValueError(
        "Unexpected ACD011/ACD040 overlap."
    )


if overlap_011_110 != 0:

    raise ValueError(
        "Unexpected ACD011/ACD110 overlap."
    )


if overlap_040_110 != 0:

    raise ValueError(
        "Unexpected ACD040/ACD110 overlap."
    )


if no_language_response_count != 25:

    raise ValueError(
        "Unexpected all-language-fields-missing count."
    )


print(
    "PASS: ACQ question-group structure "
    "matches the attached release."
)


# ============================================================
# 20. FINAL ZERO CHECK BEFORE EXPORT
# ============================================================

#
# Source XPT contained:
#
#     tiny zeros = 0
#     ordinary zeros = 0
#
#
# Therefore processing must not create any zeros.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO CHECK BEFORE EXPORT ---"
)

print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected numeric zeros were created:\n"
        + str(
            final_zero_counts
        )
    )


print(
    "PASS: No numeric zeros created."
)


# ============================================================
# 21. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# ACQ_J required NO numerical repairs.
#
# Therefore every value must remain numerically
# identical to the original XPT.
#
# Converting:
#
#     93705.0 -> 93705
#
# changes text formatting only.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: All 6 variables remain numerically "
    "identical to source XPT."
)


# ============================================================
# 22. STORE MISSING COUNTS
# ============================================================

expected_missing_counts = {
    col: int(
        expected_df[col]
        .isna()
        .sum()
    )
    for col in EXPECTED_COLUMNS
}


# ============================================================
# 23. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
#
# ACQ_J itself has no genuine decimals.
#
# For other NHANES datasets containing genuine
# continuous decimals, continue preserving them.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 24. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 25. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all 6 ACQ_J variables."
)


# ============================================================
# 26. MISSING-VALUE FIDELITY
# ============================================================

missing_position_mismatches = 0

missing_count_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_position_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


    actual_missing_count = int(
        roundtrip_df[col]
        .isna()
        .sum()
    )


    if (
        actual_missing_count
        !=
        expected_missing_counts[col]
    ):

        missing_count_failures.append(
            col
        )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_position_mismatches
)


if missing_position_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


if missing_count_failures:

    raise ValueError(
        "Missing counts changed in:\n"
        + str(
            missing_count_failures
        )
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 27. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 28. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 29. RAW CSV .0 CHECK
# ============================================================

#
# ACQ_J contains NO genuine decimal fields.
#
# Therefore checking all six fields for unnecessary
# ".0" is safe for this specific file.
#

dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


print(
    "\n--- RAW CSV .0 CHECK ---"
)

print(
    dot_zero_errors
)


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' "
    "in any ACQ_J field."
)


# ============================================================
# 30. FINAL ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    count = int(
        values
        .eq(0)
        .sum()
    )


    if count > 0:

        csv_zero_counts[col] = count


print(
    "\n--- FINAL CSV ZERO CHECK ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected numeric zeros found:\n"
        + str(
            csv_zero_counts
        )
    )


print(
    "PASS: Final CSV contains no numeric zeros."
)


# ============================================================
# 31. FINAL FREQUENCY CHECKS
# ============================================================

FINAL_FREQUENCIES = {

    "ACD011A": {
        1: 5230
    },

    "ACD011B": {
        8: 16
    },

    "ACD011C": {
        9: 159
    },

    "ACD040": {
        1: 529,
        2: 322,
        3: 334,
        4: 353,
        5: 414,
        7: 2,
        9: 3
    },

    "ACD110": {
        1: 439,
        2: 107,
        3: 124,
        4: 134,
        5: 327
    }
}


print(
    "\n--- FINAL CSV FREQUENCY CHECK ---"
)


for col, expected_freq in FINAL_FREQUENCIES.items():

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    actual_freq = (
        values
        .dropna()
        .astype(int)
        .value_counts()
        .sort_index()
        .to_dict()
    )


    print(
        f"{col}:",
        actual_freq
    )


    if actual_freq != expected_freq:

        raise ValueError(
            f"Final {col} frequencies changed."
        )


print(
    "PASS: Final CDC frequency distributions preserved."
)


# ============================================================
# 32. FINAL GROUP-OVERLAP CHECK
# ============================================================

final_acd011 = (
    raw_csv_df[
        [
            "ACD011A",
            "ACD011B",
            "ACD011C"
        ]
    ]
    .ne("")
    .any(axis=1)
    .to_numpy(
        dtype=bool
    )
)


final_acd040 = (
    raw_csv_df["ACD040"]
    .ne("")
    .to_numpy(
        dtype=bool
    )
)


final_acd110 = (
    raw_csv_df["ACD110"]
    .ne("")
    .to_numpy(
        dtype=bool
    )
)


final_overlap_011_040 = int(
    np.sum(
        final_acd011
        &
        final_acd040
    )
)


final_overlap_011_110 = int(
    np.sum(
        final_acd011
        &
        final_acd110
    )
)


final_overlap_040_110 = int(
    np.sum(
        final_acd040
        &
        final_acd110
    )
)


print(
    "\n--- FINAL QUESTION-GROUP OVERLAP ---"
)

print(
    "ACD011 / ACD040:",
    final_overlap_011_040
)

print(
    "ACD011 / ACD110:",
    final_overlap_011_110
)

print(
    "ACD040 / ACD110:",
    final_overlap_040_110
)


if final_overlap_011_040 != 0:

    raise ValueError(
        "Final ACD011/ACD040 overlap changed."
    )


if final_overlap_011_110 != 0:

    raise ValueError(
        "Final ACD011/ACD110 overlap changed."
    )


if final_overlap_040_110 != 0:

    raise ValueError(
        "Final ACD040/ACD110 overlap changed."
    )


print(
    "PASS: Final question-group separation preserved."
)


# ============================================================
# 33. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 34. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93705:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found in final CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 35. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "ACQ_J CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2017-2018"
)


print(
    "Component: Questionnaire - Acculturation"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny XPORT artifacts:",
    tiny_total
)


print(
    "Original ordinary numeric zeros:",
    ordinary_zero_total
)


print(
    "Validated zero restorations:",
    "NONE"
)


print(
    "Variables with genuine fractional values:",
    fractional_columns
)


print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


print(
    "ACD011 response-group participants:",
    f"{acd011_count:,}"
)


print(
    "ACD040 Hispanic-language responses:",
    f"{acd040_count:,}"
)


print(
    "ACD110 Asian-language responses:",
    f"{acd110_count:,}"
)


print(
    "All language-response variables missing:",
    no_language_response_count
)


print(
    "Question-group overlaps:",
    (
        overlap_011_040
        +
        overlap_011_110
        +
        overlap_040_110
    )
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0':",
    "0 expected"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "ACQ_J requires NO tiny-zero restoration."
)


print(
    "All six fields contain only whole-number "
    "IDs/categorical codes."
)


print(
    "All six can therefore be exported as nullable Int64 "
    "without rounding."
)


print(
    "No df.round() and no float_format are used."
)


print(
    "Your broader NHANES fidelity rule remains unchanged:"
)


print(
    "Genuine Dietary and other continuous decimal "
    "values remain exactly AS IS."
)

NHANES 2017-2018 ACQ_J XPT -> CSV
Rows: 8,421
Columns: 6
PASS: Structure = 8,421 rows x 6 columns.

--- FIELD TYPE PLAN ---
Character fields: []
Genuine decimal fields: []
Whole-number/code fields: ['SEQN', 'ACD011A', 'ACD011B', 'ACD011C', 'ACD040', 'ACD110']

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Variables containing tiny artifacts: 0
Ordinary numeric zeros: 0
PASS: ACQ_J contains 0 tiny artifacts and 0 ordinary numeric zeros.

PASS: No tiny-to-zero repair required.

--- FRACTIONAL VALUE DETECTION ---
SEQN: 0 fractional observations
ACD011A: 0 fractional observations
ACD011B: 0 fractional observations
ACD011C: 0 fractional observations
ACD040: 0 fractional observations
ACD110: 0 fractional observations
PASS: No genuine decimal/fractional variables found in ACQ_J.

--- SEQN BEFORE CLEANING ---
Minimum: 93705.0
Maximum: 102956.0
Missing: 0
Duplicates: 0
PASS: SEQN 93705-102956 validated.
PASS: All 6 ACQ_J variables are mathematically whole-number.
PASS: All 6 fields co